In [1]:
import os
import pandas as pd
import psycopg2
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["IPSTACK_KEY"] = "d46578cdac45440dde5d083221f927ff"

db_name = os.getenv("DB_NAME")
print(f"Zaladowano konfiguracje dla bazy: {db_name}")
print(f"Klucz API obecny: {bool(os.environ.get('IPSTACK_KEY'))}")

Zaladowano konfiguracje dla bazy: Soc
Klucz API obecny: True


In [2]:
class SentinelDB:
    def __init__(self):
        self.conn_params = {
            "host": os.getenv("DB_HOST"),
            "database": os.getenv("DB_NAME"),
            "user": os.getenv("DB_USER"),
            "password": os.getenv("DB_PASS"),
            "port": os.getenv("DB_PORT")
        }
        self._init_db()

    def _get_connection(self):
        return psycopg2.connect(**self.conn_params)

    def _init_db(self):
        query = """
        CREATE TABLE IF NOT EXISTS incidents (
            id SERIAL PRIMARY KEY,
            ip_address VARCHAR(45),
            country VARCHAR(100),
            status_code INTEGER,
            path TEXT,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        """
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(query)
            conn.commit()
        print(f"Baza danych gotowa.")

    def save_alert(self, data):
        query = """
            INSERT INTO incidents (ip_address, country, status_code, path)
            VALUES (%s, %s, %s, %s)
        """
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(query, (
                    data['ip'], 
                    data.get('country', 'Unknown'), 
                    data.get('status', 403), 
                    data.get('path', '/login')
                ))
            conn.commit()

In [3]:
import re
import requests

class LogParser:
    def parse(self, file_path):
        logs = []
        try:
            with open(file_path, 'r') as f:
                for line in f:
                    ip_match = re.search(r'(\d+\.\d+\.\d+\.\d+)', line)
                    if ip_match:
                        logs.append({'ip': ip_match.group(1)})
        except FileNotFoundError:
            print(f"Błąd: Plik {file_path} nie został znaleziony.")
        return logs

class ThreatEnricher:
    def __init__(self):
        self.api_key = os.getenv("IPSTACK_KEY")

    def get_country(self, ip):
        if not self.api_key:
            return "Brak API Key"
        try:
            url = f"http://api.ipstack.com/{ip}?access_key={self.api_key}"
            response = requests.get(url, timeout=5).json()
            return response.get('country_name', 'Unknown')
        except:
            return "Unknown"

class CorrelationEngine:
    def __init__(self, db):
        self.db = db

    def get_brute_force_report(self, threshold=1):
        query = f"""
            SELECT ip_address, country, COUNT(*) as attempts
            FROM incidents
            GROUP BY ip_address, country
            HAVING COUNT(*) >= {threshold}
            ORDER BY attempts DESC;
        """
        with self.db._get_connection() as conn:
            return pd.read_sql_query(query, conn)

In [5]:
with db._get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE incidents;")
    conn.commit()
print("Baza danych została wyczyszczona.")

Baza danych została wyczyszczona.


In [7]:
def send_discord_alert(ip, country, attempts):
    webhook_url = os.getenv("DISCORD_WEBHOOK_URL")
    if webhook_url:
        message = {
            "content": f"🚨 **DETEKCJA SIEM: ATAK BRUTE-FORCE** 🚨\n"
                       f"**IP:** `{ip}`\n"
                       f"**Kraj:** {country}\n"
                       f"**Liczba incydentów:** `{attempts}`\n"
                       f"**Status:** WYKRYTO ZAGROŻENIE"
        }
        requests.post(webhook_url, json=message, timeout=5)

db = SentinelDB()
parser = LogParser()
enricher = ThreatEnricher()
engine = CorrelationEngine(db)

log_entries = parser.parse('serwer.log')

for entry in log_entries:
    entry['country'] = enricher.get_country(entry['ip'])
    db.save_alert(entry)
    print(f"Zapisano: {entry['ip']} ({entry['country']})")

threshold = 3
report = engine.get_brute_force_report(threshold=threshold)

if not report.empty:
    for _, row in report.iterrows():
        send_discord_alert(row['ip_address'], row['country'], row['attempts'])
    print(f"\nWysłano alerty dla {len(report)} adresów IP.")
else:
    print("\nBrak incydentów powyżej progu.")

display(report)

Baza danych gotowa.
Zapisano: 192.168.1.15 (None)
Zapisano: 8.8.8.8 (United States)
Zapisano: 185.12.33.9 (Unknown)
Zapisano: 185.12.33.9 (Unknown)
Zapisano: 185.12.33.9 (France)
Zapisano: 1.1.1.1 (Unknown)
Zapisano: 95.161.225.100 (Kazakhstan)
Zapisano: 192.168.1.102 (Unknown)
Zapisano: 2.57.122.1 (Unknown)
Zapisano: 8.8.8.8 (United States)
Zapisano: 77.222.54.12 (Unknown)
Zapisano: 10.0.0.5 (None)
Zapisano: 203.0.113.42 (Unknown)
Zapisano: 192.168.1.15 (None)
Zapisano: 185.12.33.9 (Unknown)
Zapisano: 185.12.33.9 (Unknown)
Zapisano: 185.12.33.9 (France)
Zapisano: 185.12.33.9 (Unknown)
Zapisano: 185.12.33.9 (France)
Zapisano: 185.12.33.9 (Unknown)


C:\Users\HP\AppData\Local\Temp\ipykernel_28580\637410430.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)



Wysłano alerty dla 3 adresów IP.


,ip_address,country,attempts
0,185.12.33.9,Unknown,8
1,185.12.33.9,France,4
2,192.168.1.15,NaN,4
